In [5]:
import pandas as pd

def calculate_wear_proxy(cof_steady_mean, cof_std, alpha=10, beta=5):
    """
    Calculate wear_proxy based on cof_steady_mean and cof_std.

    Parameters:
    - cof_steady_mean: Steady-state coefficient of friction (float)
    - cof_std: Standard deviation of coefficient of friction (float)
    - alpha: Weight for cof_steady_mean (default=10)
    - beta: Weight for cof_std (default=5)

    Returns:
    - wear_proxy: Calculated wear proxy value (float)
    """
    return 1 / (1 + alpha * cof_steady_mean + beta * cof_std)

# Load experimental data from a CSV file
input_csv_path = "../src/sft_qwen3_14b_out/R2_results_template.csv"  # 输入文件路径
output_csv_path = "../src/sft_qwen3_14b_out/R2_results_filled.csv"  # 输出文件路径

# Read the input CSV file
data = pd.read_csv(input_csv_path)

# 转换为数值，无法转换的为 NaN
data["cof_steady_mean"] = pd.to_numeric(data["cof_steady_mean"], errors="coerce")
data["cof_std"] = pd.to_numeric(data["cof_std"], errors="coerce")

# 只对能正常计算的行赋值，其他为 NaN
def safe_wear_proxy(row):
    if pd.notna(row["cof_steady_mean"]) and pd.notna(row["cof_std"]):
        return calculate_wear_proxy(row["cof_steady_mean"], row["cof_std"])
    else:
        return float("nan")

data["wear_proxy"] = data.apply(safe_wear_proxy, axis=1)

# Calculate wear_proxy for each candidate
data["wear_proxy"] = data.apply(
    lambda row: calculate_wear_proxy(row["cof_steady_mean"], row["cof_std"]),
    axis=1
)

# Save the results to a new CSV file
data.to_csv(output_csv_path, index=False)
print(f"Results saved to {output_csv_path}")
print(data)

Results saved to ../src/sft_qwen3_14b_out/R2_results_filled.csv
  candidate_id  cof_steady_mean  cof_std  wear_proxy  compression_modulus_MPa  \
0        R2-01           0.0227   0.0282    0.730994                 3.665052   
1        R2-02              NaN      NaN         NaN                      NaN   
2        R2-03              NaN      NaN         NaN                      NaN   
3        R2-04           0.1223   0.0313    0.420256                 1.853742   
4        R2-05              NaN      NaN         NaN                 2.099337   
5        R2-06           0.0926   0.0159    0.498629                 5.107006   
6        R2-07              NaN      NaN         NaN                      NaN   
7        R2-08              NaN      NaN         NaN                      NaN   

    failure_type  failure_time_min  \
0           none               NaN   
1  cant_hold_10N               NaN   
2     not_gelled               NaN   
3           none               NaN   
4           none